In [1]:
# ============================================================================
# NB01_data_preparation.ipynb
# Study 2: Uncovering Hidden Evaluator Judgment Patterns in Credit Rating
# Overrides — Nested Cross-Validation Version
#
# ----------------------------------------------------------------------------
# WHY THIS NOTEBOOK LOOKS DIFFERENT FROM THE ORIGINAL VERSION
# ----------------------------------------------------------------------------
# The original version fit the system credit grading model, assigned grades,
# and simulated overrides ONCE on the full N = 43,405 sample, then split the
# resulting upgrade cohort for downstream analysis. That approach allows
# information leakage: firms held out for later factor-validation would
# already have influenced the system model, grade thresholds, and override
# probabilities that were applied to the training firms (and vice versa).
#
# This version implements a full NESTED design:
#
#   1. An OUTER 5-fold split is defined first, on the full N = 43,405 sample.
#   2. For EACH outer fold f in {0, 1, 2, 3, 4}:
#        a. The system credit grading model (Section 5) is trained using
#           ONLY the outer-training firms of fold f.
#        b. Grades (Section 6) are assigned to ALL firms (train + holdout)
#           using thresholds derived ONLY from outer-training firms.
#        c. The override simulation (Section 7) is applied to all firms
#           using the same fixed random seed and probability tables
#           (these are analyst-specified constants, not fitted parameters,
#           so applying them uniformly does not leak information).
#        d. The upgrade cohort (Section 8) is extracted and tagged with
#           fold f and with an `outer_split` label of "train" or "holdout".
#   3. The five per-fold master/cohort files are saved separately.
#
# Downstream notebooks (NB03-NB06) will read only the "train" partition of
# a given fold to DISCOVER warning factors. NB07 will read only the
# "holdout" partition of the SAME fold to VALIDATE those factors. Because
# steps (a)-(c) above are refit per fold using only that fold's training
# firms, no holdout firm influences any part of the pipeline applied to it.
#
# ----------------------------------------------------------------------------
# OUTPUTS
# ----------------------------------------------------------------------------
#   data/processed/fold_{f}/polish_master_fold{f}.parquet   (f = 0..4)
#   data/processed/fold_{f}/upgrade_cohort_fold{f}.parquet  (f = 0..4)
#   data/processed/fold_assignment_summary.csv
#
# Each polish_master_fold{f}.parquet contains ALL 43,405 rows, with:
#   - outer_split == "train"   for firms used to fit fold f's system model
#   - outer_split == "holdout" for firms held out in fold f
# and grade_ordinal / system_grade / grade_diff / override_flag columns
# that were computed using ONLY fold f's training firms.
#
# Download data: https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data
# Place 1year.arff ~ 5year.arff in data/raw/ before running.
# ============================================================================

import os
import warnings
import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

RAW_DIR      = "../data/raw/"
PROC_DIR     = "../data/processed/"
RANDOM_SEED  = 42
N_OUTER_FOLDS = 5

os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)
np.random.seed(RANDOM_SEED)

print("Directories ready.")


# ── 1. Load .arff files ───────────────────────────────────────────────────────

ARFF_FILES = {
    f"{i}year": os.path.join(RAW_DIR, f"{i}year.arff")
    for i in range(1, 6)
}

frames = []
for year_key, fpath in ARFF_FILES.items():
    if not os.path.exists(fpath):
        print(f"[SKIP] {fpath} not found.")
        continue
    data, meta = arff.loadarff(fpath)
    df = pd.DataFrame(data)
    df = df.applymap(lambda x: x.decode() if isinstance(x, bytes) else x)
    df["year_horizon"] = int(year_key[0])
    frames.append(df)
    print(f"Loaded {year_key}: {len(df):,} rows, {df.shape[1]} cols")

if not frames:
    raise FileNotFoundError(
        "No .arff files found. Download from UCI and place in data/raw/"
    )

master_raw = pd.concat(frames, ignore_index=True)
print(f"\nMaster shape: {master_raw.shape}")


# ── 2. Target variable & column renaming ─────────────────────────────────────

master_raw["default"] = (
    master_raw["class"].astype(str).str.strip().isin(["1", "1.0", "b'1'"])
).astype(int)

rename_map = {
    "Attr1" : "net_profit_to_assets",
    "Attr2" : "total_liabilities_to_assets",
    "Attr3" : "working_capital_to_assets",
    "Attr4" : "current_assets_to_short_liabilities",
    "Attr5" : "cash_to_current_liabilities",
    "Attr6" : "retained_earnings_to_assets",
    "Attr7" : "ebit_to_assets",
    "Attr8" : "book_value_equity_to_liabilities",
    "Attr9" : "sales_to_assets",
    "Attr10": "equity_to_assets",
}
master_raw.rename(columns=rename_map, inplace=True)

print(f"Overall default rate: {master_raw['default'].mean():.4f}  ({master_raw['default'].mean()*100:.2f}%)")


# ── 3. Outlier clipping (1st / 99th percentile) ───────────────────────────────
#
# NOTE ON LEAKAGE: clipping thresholds here are computed on the FULL sample,
# not per outer fold. This is a deliberate, documented simplification: the
# 1st/99th percentile clip bounds are a fixed data-cleaning convention (not
# a fitted model parameter that encodes label information), and recomputing
# them per fold would only shift a small number of extreme values at the
# margins. This choice should be stated explicitly in the paper's methods
# section as a minor, non-label-derived preprocessing step applied uniformly.

numeric_cols = master_raw.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ("default", "year_horizon")]

for col in numeric_cols:
    lo, hi = master_raw[col].quantile([0.01, 0.99])
    master_raw[col] = master_raw[col].clip(lo, hi)

print(f"Clipped {len(numeric_cols)} numeric columns (bounds computed on full sample; see note above).")


# ── 4. Missing value imputation ───────────────────────────────────────────────
#
# NOTE ON LEAKAGE: as with clipping above, median imputation here uses the
# full-sample median rather than a per-fold median. This is the same
# documented simplification as Section 3. If a reviewer requires strict
# per-fold imputation, this section would need to move inside the per-fold
# loop in Section 5 and use only outer-training-firm medians — flagged here
# as a possible follow-up refinement, not implemented in this version.

missing     = master_raw.isnull().sum()
missing_pct = (missing / len(master_raw) * 100).round(2)
missing_report = pd.DataFrame({"missing_n": missing, "missing_pct": missing_pct})
missing_report = (
    missing_report[missing_report["missing_n"] > 0]
    .sort_values("missing_pct", ascending=False)
)
print(f"Columns with missing values: {len(missing_report)}")

for col in numeric_cols:
    if master_raw[col].isnull().any():
        master_raw[col].fillna(master_raw[col].median(), inplace=True)

print(f"Remaining missing values: {master_raw.isnull().sum().sum()}")


# ── 4b. [NEW] Outer stratified 5-fold split ───────────────────────────────────
#
# This fold assignment is created ONCE, on the full cleaned sample, before
# any label-derived model fitting happens. It governs which firms are
# "outer-training" (used to fit the system model, grade thresholds, and
# hence used for factor DISCOVERY in NB03-NB06) versus "outer-holdout"
# (used only for factor VALIDATION in NB07) within each of the 5 folds.
#
# Stratification variable: `default` x `year_horizon` jointly. Stratifying
# on default alone would balance the overall 4.82% default rate across
# folds, but could let the year_horizon composition drift. Since forecast
# horizon affects default rate substantially (Table 1 in the paper: Year 1
# vs Year 5 differ), we stratify on the combination to keep both balanced.

strat_key = (
    master_raw["default"].astype(str) + "_" + master_raw["year_horizon"].astype(str)
)

outer_skf = StratifiedKFold(
    n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED
)

master_raw["outer_fold"] = -1
for fold_id, (_, holdout_idx) in enumerate(outer_skf.split(master_raw, strat_key)):
    master_raw.loc[master_raw.index[holdout_idx], "outer_fold"] = fold_id

assert (master_raw["outer_fold"] == -1).sum() == 0, "Some rows were not assigned an outer fold."

print(f"\nOuter fold assignment ({N_OUTER_FOLDS}-fold, stratified on default x year_horizon):")
print(master_raw["outer_fold"].value_counts().sort_index())
print("\nDefault rate by outer fold (reference, full sample: {:.4f}):".format(master_raw["default"].mean()))
print(master_raw.groupby("outer_fold")["default"].mean().round(4))


# ── 5-8. [NEW] Per-fold nested pipeline ───────────────────────────────────────
#
# For each outer fold f:
#   - "outer-training" firms  = all firms where outer_fold != f
#   - "outer-holdout"  firms  = all firms where outer_fold == f
#
# The system model (Section 5) is fit using ONLY outer-training firms.
# Grade thresholds (Section 6) are computed from the outer-training firms'
# predicted probabilities, then APPLIED to all firms (train + holdout) using
# those training-derived thresholds. This means holdout firms get a grade,
# but the grade boundaries were never informed by holdout labels.
#
# The override simulation (Section 7) uses fixed, literature-motivated
# probability tables (UPGRADE_PROB / DOWNGRADE_PROB) and a fixed random seed
# — these are simulation design choices, not fitted parameters, so no
# leakage is introduced by applying them identically across train/holdout.

FEATURE_COLS = [c for c in numeric_cols if c != "default"]

GRADE_LABELS = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]
GRADE_NUM    = {"AAA": 7, "AA": 6, "A": 5, "BBB": 4, "BB": 3, "B": 2, "CCC": 1}
INV_GRADE_NUM = {v: k for k, v in GRADE_NUM.items()}

UPGRADE_PROB = {
    "CCC": 0.80, "B": 0.70, "BB": 0.50,
    "BBB": 0.35, "A": 0.20, "AA": 0.00, "AAA": 0.00,
}
DOWNGRADE_PROB = {
    "CCC": 0.00, "B": 0.10, "BB": 0.15,
    "BBB": 0.20, "A": 0.25, "AA": 0.30, "AAA": 0.35,
}

fold_summary_rows = []

for f in range(N_OUTER_FOLDS):

    print(f"\n{'='*76}\nOUTER FOLD {f}\n{'='*76}")

    fold_df = master_raw.copy()
    fold_df["outer_split"] = np.where(
        fold_df["outer_fold"] == f, "holdout", "train"
    )

    train_mask = fold_df["outer_split"] == "train"
    holdout_mask = fold_df["outer_split"] == "holdout"

    print(f"  Outer-training firms : {train_mask.sum():,}")
    print(f"  Outer-holdout firms  : {holdout_mask.sum():,}")

    # -- Section 5 (per-fold): fit system model on outer-training firms only --
    #
    # A StandardScaler is also fit on outer-training firms only, then applied
    # to both train and holdout rows, so holdout feature scaling does not
    # leak holdout distributional information into the scaler.

    X_train = fold_df.loc[train_mask, FEATURE_COLS].values
    y_train = fold_df.loc[train_mask, "default"].values
    X_all   = fold_df[FEATURE_COLS].values

    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_all_scaled   = scaler.transform(X_all)

    lr_fold = LogisticRegression(
        C=0.1, class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED
    )
    lr_fold.fit(X_train_scaled, y_train)

    # pd_system for ALL firms (train + holdout), but the model itself was
    # fit using only train firms.
    fold_df["pd_system"] = lr_fold.predict_proba(X_all_scaled)[:, 1]

    train_auc = roc_auc_score(y_train, fold_df.loc[train_mask, "pd_system"])
    holdout_auc = roc_auc_score(
        fold_df.loc[holdout_mask, "default"], fold_df.loc[holdout_mask, "pd_system"]
    )
    print(f"  System model ROC-AUC (outer-train, in-sample) : {train_auc:.4f}")
    print(f"  System model ROC-AUC (outer-holdout, true OOS): {holdout_auc:.4f}")

    # -- Section 6 (per-fold): grade thresholds from outer-training firms only --
    #
    # Quantile cut points are computed using ONLY outer-training pd_system
    # values, then applied to classify ALL firms (train + holdout) into the
    # same 7 grade buckets. Holdout firms are graded using train-derived
    # boundaries, never their own distribution.

    train_pd = fold_df.loc[train_mask, "pd_system"].values
    quantile_cuts = np.quantile(train_pd, np.linspace(0, 1, 8))
    quantile_cuts[0]  = -np.inf
    quantile_cuts[-1] =  np.inf

    fold_df["system_grade"] = pd.cut(
        fold_df["pd_system"],
        bins=quantile_cuts,
        labels=GRADE_LABELS,
        include_lowest=True
    )
    fold_df["grade_ordinal"] = fold_df["system_grade"].map(GRADE_NUM).astype(int)

    print("  Grade distribution (outer-training firms, target ~balanced):")
    print(
        fold_df.loc[train_mask, "system_grade"]
        .value_counts()
        .sort_index()
        .to_string()
        .replace("\n", "\n    ")
    )

    dr_by_grade_train = (
        fold_df.loc[train_mask]
        .groupby("system_grade", observed=True)["default"]
        .mean()
        .reindex(GRADE_LABELS)
    )
    monotonic_ok = list(dr_by_grade_train) == sorted(dr_by_grade_train)
    print(f"  [Check] Default rates monotonic AAA→CCC on outer-training firms: {monotonic_ok}")
    if not monotonic_ok:
        print("    [WARNING] Non-monotonic default rates within this fold's training grades.")

    # -- Section 7 (per-fold): override simulation, identical rule set --
    #
    # UPGRADE_PROB / DOWNGRADE_PROB and RANDOM_SEED are fixed constants
    # (not fitted from data), so applying them identically to train and
    # holdout firms within this fold does not leak label information.
    # A fold-specific rng stream (seeded from RANDOM_SEED + f) is used so
    # results are reproducible per fold without reusing identical draws
    # across folds.

    rng_fold = np.random.default_rng(RANDOM_SEED + f)
    grade_diffs = []
    for grade in fold_df["system_grade"].astype(str):
        g_num  = GRADE_NUM.get(grade, 4)
        up_p   = UPGRADE_PROB.get(grade, 0.0)
        down_p = DOWNGRADE_PROB.get(grade, 0.0)
        r      = rng_fold.random()
        if r < up_p and g_num < 7:
            grade_diffs.append(-1)
        elif r < up_p + down_p and g_num > 1:
            grade_diffs.append(1)
        else:
            grade_diffs.append(0)

    fold_df["grade_diff"]          = np.array(grade_diffs, dtype=int)
    fold_df["override_flag"]       = (fold_df["grade_diff"] != 0).astype(int)
    fold_df["final_grade_ordinal"] = fold_df["grade_ordinal"].astype(int) - fold_df["grade_diff"]

    direction_map = {-1: "Upgrade", 0: "Maintain", 1: "Downgrade"}
    override_summary = (
        fold_df.groupby(fold_df["grade_diff"].map(direction_map))["default"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "default_rate", "count": "n"})
        .round(4)
    )
    print("  Default rate by override direction (full fold_df, train+holdout):")
    print(override_summary.to_string().replace("\n", "\n    "))

    # -- Section 8 (per-fold): cohort extraction --

    fold_df["from_grade"] = fold_df["grade_ordinal"].map(INV_GRADE_NUM)
    fold_df["to_grade"]   = fold_df["final_grade_ordinal"].map(INV_GRADE_NUM)
    fold_df["transition"] = fold_df["from_grade"] + "→" + fold_df["to_grade"]
    fold_df["high_risk_transition"] = (
        fold_df["from_grade"].isin(["CCC", "B"])
    ).astype(int)

    upgrade_cohort_fold = fold_df[fold_df["grade_diff"] == -1].copy()
    upgrade_cohort_fold["group"] = np.where(
        upgrade_cohort_fold["default"] == 1, "risky_upgrade", "safe_upgrade"
    )
    upgrade_cohort_fold["is_risky"] = (
        upgrade_cohort_fold["group"] == "risky_upgrade"
    ).astype(int)

    cohort_train_mask   = upgrade_cohort_fold["outer_split"] == "train"
    cohort_holdout_mask = upgrade_cohort_fold["outer_split"] == "holdout"

    risky_train_rate = upgrade_cohort_fold.loc[cohort_train_mask, "is_risky"].mean() * 100
    risky_holdout_rate = upgrade_cohort_fold.loc[cohort_holdout_mask, "is_risky"].mean() * 100

    print(f"  Upgrade cohort size (train) : {cohort_train_mask.sum():,}  |  risky_upgrade rate: {risky_train_rate:.2f}%")
    print(f"  Upgrade cohort size (holdout): {cohort_holdout_mask.sum():,}  |  risky_upgrade rate: {risky_holdout_rate:.2f}%")

    fold_dir = os.path.join(PROC_DIR, f"fold_{f}")
    os.makedirs(fold_dir, exist_ok=True)

    master_fold_path = os.path.join(fold_dir, f"polish_master_fold{f}.parquet")
    fold_df.to_parquet(master_fold_path, index=False)

    cohort_fold_path = os.path.join(fold_dir, f"upgrade_cohort_fold{f}.parquet")
    upgrade_cohort_fold.to_parquet(cohort_fold_path, index=False)

    print(f"  Saved: {master_fold_path}")
    print(f"  Saved: {cohort_fold_path}")

    fold_summary_rows.append({
        "outer_fold": f,
        "n_train_firms": int(train_mask.sum()),
        "n_holdout_firms": int(holdout_mask.sum()),
        "system_auc_train": round(train_auc, 4),
        "system_auc_holdout": round(holdout_auc, 4),
        "grades_monotonic_on_train": monotonic_ok,
        "n_upgrade_cohort_train": int(cohort_train_mask.sum()),
        "n_upgrade_cohort_holdout": int(cohort_holdout_mask.sum()),
        "risky_upgrade_rate_train_pct": round(risky_train_rate, 2),
        "risky_upgrade_rate_holdout_pct": round(risky_holdout_rate, 2),
    })


# ── 9. Cross-fold summary and balance check ───────────────────────────────────

fold_summary = pd.DataFrame(fold_summary_rows)
summary_path = os.path.join(PROC_DIR, "fold_assignment_summary.csv")
fold_summary.to_csv(summary_path, index=False)

print(f"\n{'='*76}\nCROSS-FOLD SUMMARY\n{'='*76}")
print(fold_summary.to_string(index=False))
print(f"\nSaved: {summary_path}")

max_dev = (fold_summary["risky_upgrade_rate_holdout_pct"] - 8.34).abs().max()
print(f"\nMax deviation of holdout risky_upgrade rate from original full-sample rate (8.34%): {max_dev:.2f} pp")
if max_dev > 1.5:
    print(
        "  [WARNING] risky_upgrade rate varies by more than 1.5pp across "
        "holdout folds. This is expected to some degree in a nested design "
        "since the system model is refit per fold, but a large deviation "
        "may warrant inspection (e.g., checking whether a specific fold's "
        "training partition produced unusually shifted grade thresholds)."
    )
else:
    print("  [OK] risky_upgrade rate is reasonably balanced across holdout folds.")

print("\nAUC stability across folds:")
print(f"  Outer-training AUC : mean {fold_summary['system_auc_train'].mean():.4f}, "
      f"std {fold_summary['system_auc_train'].std():.4f}")
print(f"  Outer-holdout  AUC : mean {fold_summary['system_auc_holdout'].mean():.4f}, "
      f"std {fold_summary['system_auc_holdout'].std():.4f}")

print("\nNext step -> NB02_eda.ipynb (run once per fold, on the 'train' partition "
      "of each fold_{f}/upgrade_cohort_fold{f}.parquet file)")

Directories ready.
Loaded 1year: 7,027 rows, 66 cols
Loaded 2year: 10,173 rows, 66 cols
Loaded 3year: 10,503 rows, 66 cols
Loaded 4year: 9,792 rows, 66 cols
Loaded 5year: 5,910 rows, 66 cols

Master shape: (43405, 66)
Overall default rate: 0.0482  (4.82%)
Clipped 64 numeric columns (bounds computed on full sample; see note above).
Columns with missing values: 64
Remaining missing values: 0

Outer fold assignment (5-fold, stratified on default x year_horizon):
outer_fold
0    8681
1    8681
2    8681
3    8681
4    8681
Name: count, dtype: int64

Default rate by outer fold (reference, full sample: 0.0482):
outer_fold
0    0.0482
1    0.0483
2    0.0482
3    0.0482
4    0.0482
Name: default, dtype: float64

OUTER FOLD 0
  Outer-training firms : 34,724
  Outer-holdout firms  : 8,681
  System model ROC-AUC (outer-train, in-sample) : 0.7958
  System model ROC-AUC (outer-holdout, true OOS): 0.7625
  Grade distribution (outer-training firms, target ~balanced):
system_grade
    AAA    4961
   